<a href="https://colab.research.google.com/github/RafihaikalP/BigData26_A_2411532002_Rafi-Haikal-Pratama/blob/main/Praktikum5/BD_P05_LATIHAN_2411532002_RafiHaikalPratama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 5 Big Data — Lanjutan: Latihan, Tugas Mandiri, Analisis Hasil, Studi Kasus, dan Evaluasi
Notebook ini melanjutkan `Praktikum_5_Big_Data.ipynb`. Semua angka pada jawaban diambil dari output notebook utama Anda (waktu dari Colab Anda; jumlah dan proporsi dari data yang sama).

**Prasyarat:** notebook utama sudah dijalankan sampai H-13, sehingga folder `lapisan_terkurasi/`, `lapisan_mentah/`, dan `detik_news_title.csv` ada di folder kerja di Google Drive.

## 0. Persiapan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/BigData/Praktikum5")
print("Folder kerja:", os.getcwd())

Mounted at /content/drive
Folder kerja: /content/drive/MyDrive/BigData/Praktikum5


In [ ]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

In [ ]:
import os, re, json, time, zlib, collections
from pathlib import Path
import pandas as pd

TERKURASI = "lapisan_terkurasi"
splits = sorted(Path(TERKURASI).rglob("*.parquet"))

STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

print("Jumlah input split:", len(splits))

Jumlah input split: 9


## N. Latihan
### Latihan 1 — Stopword domain
Lima kata ditambahkan ke `STOPWORD`: **rp, ri, the, foto, galeri**. Alasannya: `rp`, `ri`, dan `the` muncul di lima kata teratas (modul menyebutnya), sedangkan `foto` dan `galeri` adalah kata penanda format konten (rubrik foto/galeri), bukan topik berita.

In [ ]:
TAMBAHAN = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_DOMAIN = STOPWORD | TAMBAHAN

def mapper_kategori_sw(path, stop):
    """Sama seperti mapper_kategori di H-13, tetapi stopword dapat diganti."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in stop:
                c[(kategori, kata)] += 1
    return c

def top_per_kategori(stop, n=5):
    gabung = collections.Counter()
    for c in (mapper_kategori_sw(p, stop) for p in splits):
        gabung.update(c)
    per = collections.defaultdict(list)
    for (kat, kata), j in gabung.items():
        per[kat].append((j, kata))
    return {kat: sorted(v, reverse=True)[:n] for kat, v in per.items()}

# jumlah kemunculan kelima kata tambahan (tanpa stopword apa pun)
total_kata = collections.Counter()
for p in splits:
    total_kata.update(mapper_combiner(p))
print("Kemunculan kata yang ditambahkan:", {k: total_kata[k] for k in sorted(TAMBAHAN)})

sebelum = top_per_kategori(STOPWORD)
sesudah = top_per_kategori(STOPWORD_DOMAIN)
for kat in sorted(sebelum):
    print(f"{kat:<8} sebelum: ", ", ".join(f"{k} ({n:,})" for n, k in sebelum[kat]))
    print(f"{'':<8} sesudah: ", ", ".join(f"{k} ({n:,})" for n, k in sesudah[kat]))

Kemunculan kata yang ditambahkan: {'foto': 1295, 'galeri': 450, 'ri': 2206, 'rp': 3154, 'the': 678}
finance  sebelum:  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
         sesudah:  corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)
food     sebelum:  makanan (699), makan (539), foto (458), galeri (438), enak (358)
         sesudah:  makanan (699), makan (539), enak (358), bikin (343), rumah (311)
health   sebelum:  corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
         sesudah:  corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      sebelum:  corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
         sesudah:  corona (1,658), anak (793), rumah (561), meninggal (545), film (489)
inet     sebelum:  corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
         sesudah:  corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     sebe

### Latihan 2 — Key per bulan: (bulan, kata)

In [ ]:
def mapper_bulan(path):
    """Key gabungan (bulan, kata) dengan combiner di dalam mapper."""
    df_ = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(df_["judul"], df_["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD_DOMAIN:
                c[(b, kata)] += 1
    return c

gabung_bulan = collections.Counter()
judul_per_bulan = collections.Counter()
for p in splits:
    gabung_bulan.update(mapper_bulan(p))
    judul_per_bulan.update(pd.read_parquet(p, columns=["bulan"])["bulan"])

per_bulan = collections.defaultdict(list)
for (b, kata), n in gabung_bulan.items():
    per_bulan[b].append((n, kata))
for b in sorted(per_bulan):
    top = sorted(per_bulan[b], reverse=True)[:8]
    print(b, f"({judul_per_bulan[b]:,} judul):", ", ".join(f"{k} ({n:,})" for n, k in top))

# Perbandingan Januari -> Mei memakai frekuensi per 1.000 judul (jumlah judul tiap bulan berbeda)
def per_seribu(b):
    return {k: n / judul_per_bulan[b] * 1000 for n, k in per_bulan[b]}
jan, mei = per_seribu("2020-01"), per_seribu("2020-05")
top_jan = {k for _, k in sorted(per_bulan["2020-01"], reverse=True)[:15]}
top_mei = {k for _, k in sorted(per_bulan["2020-05"], reverse=True)[:15]}
print("\nHanya di 15 teratas Januari:", sorted(top_jan - top_mei))
print("Hanya di 15 teratas Mei    :", sorted(top_mei - top_jan))
print("\nKata dengan kenaikan terbesar Jan -> Mei (per 1.000 judul, minimal 50 kemunculan di Mei):")
naik = sorted(((mei[k] - jan.get(k, 0), k) for k in mei if gabung_bulan[("2020-05", k)] >= 50), reverse=True)[:8]
for d, k in naik:
    print(f"  {k:<12} Jan {jan.get(k, 0):6.1f} -> Mei {mei[k]:6.1f}")

2020-01 (16,258 judul): banjir (1,220), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428), minta (427), kpk (415)
2020-02 (16,135 judul): corona (1,076), polisi (630), virus (596), banjir (572), anak (453), indonesia (415), jakarta (407), wni (404)
2020-03 (18,103 judul): corona (6,986), virus (1,340), positif (925), pasien (749), rumah (695), cegah (665), indonesia (585), jokowi (537)
2020-04 (16,636 judul): corona (4,914), positif (828), pandemi (706), psbb (689), warga (609), meninggal (577), rumah (576), kasus (557)
2020-05 (16,352 judul): corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562), psbb (522), covid (522)
2020-06 (7,334 judul): corona (943), normal (471), new (465), kasus (315), juni (262), polisi (246), baru (225), psbb (214)

Hanya di 15 teratas Januari: ['banjir', 'bikin', 'china', 'jakarta', 'jokowi', 'kpk', 'minta', 'mobil', 'tahun']
Hanya di 15 teratas Mei    : ['covid', 'lebaran', 'new', 'normal', 'orang', 'pandemi',

### Latihan 3 — Bigram

In [ ]:
def mapper_bigram(path):
    """Pasangan kata berurutan per judul, mis. ('virus', 'corona')."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    hasil = []
    for t in judul:
        w = tokenisasi(t)
        hasil.extend(zip(w, w[1:]))
    return hasil

def mapper_bigram_combiner(path):
    return collections.Counter(mapper_bigram(path))

pasangan_bigram = [mapper_bigram(p) for p in splits]
setelah_combiner = [mapper_bigram_combiner(p) for p in splits]
n_tanpa = sum(len(h) for h in pasangan_bigram)
n_dengan = sum(len(c) for c in setelah_combiner)
print("Pasangan bigram tanpa combiner :", f"{n_tanpa:,}")
print("Pasangan bigram dengan combiner:", f"{n_dengan:,}", f"(turun {n_tanpa / n_dengan:.1f} kali)")

total_bigram = collections.Counter()
for c in setelah_combiner:
    total_bigram.update(c)
print("Bigram unik:", f"{len(total_bigram):,}")
print("\n10 bigram teratas (kedua kata bukan stopword):")
for (a, b), n in [x for x in total_bigram.most_common() if x[0][0] not in STOPWORD_DOMAIN and x[0][1] not in STOPWORD_DOMAIN][:10]:
    print(f"  {a} {b:<12} {n:>6,}")

Pasangan bigram tanpa combiner : 734,783
Pasangan bigram dengan combiner: 411,217 (turun 1.8 kali)
Bigram unik: 349,114

10 bigram teratas (kedua kata bukan stopword):
  virus corona        2,915
  positif corona        1,804
  pandemi corona        1,075
  new normal          917
  pasien corona          600
  cegah corona          583
  tengah pandemi         513
  kasus positif         442
  meninggal dunia           426
  gara gara            424


### Latihan 4 — Jumlah reducer (R = 2, 8, 32)

In [ ]:
def partitioner(kata, r=4):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

hasil_map = [mapper_split(p) for p in splits]
hasil_comb = [list(mapper_combiner(p).items()) for p in splits]

baris_tabel = []
for r in (2, 4, 8, 16, 32, 64):
    b1, b2 = beban(hasil_map, r), beban(hasil_comb, r)
    adil = 100 / r
    baris_tabel.append({
        "R": r,
        "bagian adil (%)": round(adil, 2),
        "terberat tanpa combiner (%)": round(max(b1) / sum(b1) * 100, 2),
        "rasio": round(max(b1) / sum(b1) * 100 / adil, 2),
        "terberat dengan combiner (%)": round(max(b2) / sum(b2) * 100, 2),
        "rasio ": round(max(b2) / sum(b2) * 100 / adil, 2),
    })
print(pd.DataFrame(baris_tabel).to_string(index=False))

 R  bagian adil (%)  terberat tanpa combiner (%)  rasio  terberat dengan combiner (%)  rasio 
 2            50.00                        52.27   1.05                         50.23    1.00
 4            25.00                        29.17   1.17                         25.55    1.02
 8            12.50                        16.83   1.35                         12.85    1.03
16             6.25                         8.82   1.41                          6.54    1.05
32             3.12                         5.82   1.86                          3.44    1.10
64             1.56                         4.28   2.74                          1.86    1.19


### Latihan 5 — Crawl ulang dengan jeda
Portal lokal dinyalakan lagi (kode sama dengan H-2 dan H-3), lalu 50 halaman pertama di-crawl dengan jeda 0,05 detik. Waktu untuk 911 halaman dengan jeda 1 detik dihitung dari hasil ukur ini.

In [ ]:
import http.server, urllib.parse, html, threading, queue
import requests, urllib.robotparser
from bs4 import BeautifulSoup
df_sumber = pd.read_csv("detik_news_title.csv")
UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"
def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

HALAMAN_UKURAN = 100   # judul per halaman
def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN
    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):   # matikan log
            pass
        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")
    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal
server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


In [ ]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

In [ ]:
def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

In [ ]:
def ukur_crawl(n_hal, delay):
    t0 = time.perf_counter()
    n = sum(1 for _ in crawl_iter(BASE, range(1, n_hal + 1), delay=delay))
    return n, time.perf_counter() - t0

n_a, t_a = ukur_crawl(50, 0.05)   # jeda 0,05 detik
n_b, t_b = ukur_crawl(50, 0.0)    # pembanding tanpa jeda
per_halaman = t_b / 50            # biaya request + parsing per halaman (tanpa jeda)

print(f"50 halaman, jeda 0,05 s : {t_a:6.2f} detik ({n_a} judul)")
print(f"50 halaman, tanpa jeda  : {t_b:6.2f} detik ({n_b} judul)")
print(f"Selisih                 : {t_a - t_b:6.2f} detik (teori 50 x 0,05 = {50 * 0.05:.2f} detik)")

est = N_HALAMAN * (1.0 + per_halaman)
print(f"\nEstimasi 911 halaman, jeda 1 detik: {est:,.0f} detik = {est / 60:.1f} menit")
print(f"Jeda saja: 911 x 1 = 911 detik = {911 / 60:.1f} menit")

50 halaman, jeda 0,05 s :   5.65 detik (5000 judul)
50 halaman, tanpa jeda  :   3.91 detik (5000 judul)
Selisih                 :   1.73 detik (teori 50 x 0,05 = 2.50 detik)

Estimasi 911 halaman, jeda 1 detik: 982 detik = 16.4 menit
Jeda saja: 911 x 1 = 911 detik = 15.2 menit


### Latihan 6 — Sitemap lain
**Aturan etika modul tetap berlaku:** jeda 2 detik, maksimum 20 judul, dan jalankan bergiliran (bukan serentak satu kelas). Sel di bawah hanya memanggil indeks sitemap satu kali.

Alamat indeks di bawah adalah dugaan; bila modul/dosen menyebut alamat lain, ganti `SITEMAP_INDEKS`.

In [ ]:
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from urllib.parse import urlparse
POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")
def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else None,   # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

import urllib.robotparser
UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: GANTI_DENGAN_EMAIL_ANDA)"
DELAY = 2.0              # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7     # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20         # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]
_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content   # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

In [ ]:
SITEMAP_INDEKS = "https://www.detik.com/sitemap.xml"   # ganti bila alamat indeks berbeda

sesi = requests.Session()
sesi.headers["User-Agent"] = UA_DETIK
try:
    indeks = BeautifulSoup(ambil_xml(SITEMAP_INDEKS, sesi), "xml")
    daftar = [l.get_text(strip=True) for l in indeks.find_all("loc")]
    berita = [u for u in daftar if u.endswith("sitemap_news.xml")]
    print("Entri indeks:", len(daftar), "| sitemap_news:", len(berita))
    for u in berita[:20]:
        print(" ", u)
except Exception as e:
    print("Gagal membaca indeks:", type(e).__name__, "-", e)

Entri indeks: 536 | sitemap_news: 248
  https://news.detik.com/suara-pembaca/sitemap_news.xml
  https://news.detik.com/berita/sitemap_news.xml
  https://news.detik.com/internasional/sitemap_news.xml
  https://news.detik.com/bbc-world/sitemap_news.xml
  https://news.detik.com/abc-australia/sitemap_news.xml
  https://news.detik.com/dw/sitemap_news.xml
  https://news.detik.com/kolom/sitemap_news.xml
  https://news.detik.com/prokontra/sitemap_news.xml
  https://news.detik.com/pemilu/sitemap_news.xml
  https://news.detik.com/pilkada/sitemap_news.xml
  https://news.detik.com/melindungi-tuah-marwah/sitemap_news.xml
  https://news.detik.com/bangun-indonesia/sitemap_news.xml
  https://news.detik.com/nyago-bumi-sriwijaya/sitemap_news.xml
  https://inet.detik.com/news/sitemap_news.xml
  https://inet.detik.com/business/sitemap_news.xml
  https://inet.detik.com/security/sitemap_news.xml
  https://inet.detik.com/telecommunication/sitemap_news.xml
  https://inet.detik.com/law-and-policy/sitemap_news.

In [ ]:
# Pilih SATU alamat dari daftar di atas (bukan sport/basket, bukan news, bukan inet), lalu isi:
SITEMAP_BARU = "ISI_DENGAN_ALAMAT_SITEMAP_NEWS_PILIHAN_ANDA"

if SITEMAP_BARU.startswith("http"):
    SITEMAP_DIPILIH[2] = SITEMAP_BARU            # ganti sitemap ke-3 (sport/basket)
    try:
        sampel2 = crawl_detik()                  # jalankan SEKALI
    except Exception as e:
        sampel2 = []
        print("Crawling gagal:", type(e).__name__, "-", e)
else:
    sampel2 = []
    print("Isi SITEMAP_BARU terlebih dahulu.")

Isi SITEMAP_BARU terlebih dahulu.


In [ ]:
KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Sama dengan H-6."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"  : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"     : jml_kata.between(3, 30),
        "kategori dikenal"    : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"       : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"   : d["url"].fillna("").str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }

if sampel2:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap_2.jsonl", "w", encoding="utf-8") as f:
        for x in sampel2:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    df2 = pd.DataFrame(sampel2)
else:
    # tanpa internet: uji pada contoh XML modul agar aturannya tetap terlihat bekerja
    df2 = pd.DataFrame(parse_sitemap_news(CONTOH_XML))
print("Jumlah judul:", len(df2), "| kategori:", df2["kategori"].value_counts().to_dict())
for nama, lulus in aturan_kualitas(df2).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>3} dari {len(df2)}")

Jumlah judul: 3 | kategori: {'news': 3}
judul tidak kosong       gagal:   0 dari 3
judul 3-30 kata          gagal:   0 dari 3
kategori dikenal         gagal:   0 dari 3
tanggal valid            gagal:   0 dari 3
url artikel valid        gagal:   0 dari 3
judul tidak duplikat     gagal:   0 dari 3


### Latihan 7 — Kata kunci terbanyak (mapper dan reducer)
Data: `contoh_crawl/detik_sitemap.jsonl` (hasil crawl H-4). Kolom `kata_kunci` dipisah koma. Tidak semua entri sitemap punya kata kunci (pada contoh modul hanya entri pertama), jadi entri tanpa kata kunci dilewati.

In [ ]:
Path("kunci.tsv").write_text("", encoding="utf-8")
if Path("contoh_crawl/detik_sitemap.jsonl").exists():
    rekaman = [json.loads(l) for l in open("contoh_crawl/detik_sitemap.jsonl", encoding="utf-8")]
else:
    rekaman = parse_sitemap_news(CONTOH_XML)   # cadangan bila berkas crawl tidak ada
with open("kunci.tsv", "w", encoding="utf-8") as f:
    for r in rekaman:
        f.write(f"{r['kategori']}\t{r['kata_kunci']}\n")
print("Rekaman:", len(rekaman), "| punya kata kunci:", sum(1 for r in rekaman if r["kata_kunci"]))

# Versi Python: map -> combiner -> shuffle -> reduce
def map_kunci(baris):
    kategori, kk = baris.rstrip("\n").split("\t", 1)
    return [(k.strip().lower(), 1) for k in kk.split(",") if k.strip()]

pasangan = [p for baris in open("kunci.tsv", encoding="utf-8") for p in map_kunci(baris)]
shuffle_kk = collections.defaultdict(list)
for k, n in pasangan:
    shuffle_kk[k].append(n)
hasil_kk = {k: sum(v) for k, v in shuffle_kk.items()}
print("Pasangan:", len(pasangan), "| kata kunci unik:", len(hasil_kk))
for k, n in collections.Counter(hasil_kk).most_common(10):
    print(f"  {k:<30} {n}")

Rekaman: 20 | punya kata kunci: 20
Pasangan: 115 | kata kunci unik: 100
  piala dunia 2026               6
  karhutla                       3
  kebakaran hutan                3
  media sosial                   2
  medsos                         2
  komdigi                        2
  timnas spanyol                 2
  final piala dunia 2026         2
  uefa nations league            2
  silmy karim                    1


In [ ]:
# Versi Hadoop Streaming: mapper baru, reducer.py dipakai ulang dari H-12 (menjumlahkan per key)
Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')   # sama dengan reducer.py di H-12
Path("mapper_kunci.py").write_text('''import sys
for baris in sys.stdin:
    kategori, kk = baris.rstrip("\\n").split("\\t", 1)
    for k in kk.split(","):
        k = k.strip().lower()
        if k:
            print(f"{k}\\t1")
''')
!cat kunci.tsv | python3 mapper_kunci.py | LC_ALL=C sort | python3 reducer.py > hasil_kunci.txt
!sort -t$'\t' -k2,2nr hasil_kunci.txt | head -n 5

hasil_kk_pipe = {}
for baris in open("hasil_kunci.txt", encoding="utf-8"):
    k, n = baris.rstrip("\n").split("\t")
    hasil_kk_pipe[k] = int(n)
print("Hasil pipe sama dengan versi Python:", hasil_kk_pipe == hasil_kk)

piala dunia 2026	6
karhutla	3
kebakaran hutan	3
final piala dunia 2026	2
komdigi	2
Hasil pipe sama dengan versi Python: True


### Catatan hasil latihan (dari data dan jalannya di atas)
- **Latihan 1:** kelima kata tambahan (rp 3.154, ri 2.206, foto 1.295, galeri 450, the 678 kemunculan) membuat perubahan terlihat pada finance (rp dan ri keluar; masuk mau dan ekonomi), food (foto dan galeri keluar; masuk bikin dan rumah), hot ("the" keluar; masuk film), oto (rp keluar; masuk honda), dan travel (foto keluar; masuk indonesia). Health, inet, news, dan sport tidak berubah.
- **Latihan 2:** pada Januari kata teratas adalah banjir (1.220), polisi, jokowi, kpk, sedangkan pada Mei corona (2.935), kasus, positif, pandemi, lebaran, psbb, covid. Per 1.000 judul, "corona" naik dari 23,7 (Januari) menjadi 179,5 (Mei). Kata baru yang hampir tidak ada pada Januari: pandemi, psbb, covid, lebaran, new, normal.
- **Latihan 3:** bigram menghasilkan 734.783 pasangan tanpa combiner dan 411.217 dengan combiner (turun hanya 1,8 kali, jauh lebih kecil daripada 11,2 kali pada unigram, karena kombinasi dua kata jarang berulang). Bigram teratas: virus corona (2.915), positif corona (1.804), pandemi corona (1.075), new normal (917).
- **Latihan 4:** lihat tabel pada output sel; rasio beban terberat terhadap bagian adil tanpa combiner naik dari 1,05 (R = 2) menjadi 1,35 (R = 8) dan 1,86 (R = 32), sedangkan dengan combiner tetap 1,00 sampai 1,10 pada R = 2 sampai 32.
- **Latihan 5:** dengan waktu crawl Anda 41,7 detik untuk 911 halaman (sekitar 0,046 detik per halaman), 50 halaman dengan jeda 0,05 detik kira-kira 50 x (0,046 + 0,05) = 4,8 detik (ukur sendiri di sel), dan 911 halaman dengan jeda 1 detik kira-kira 911 x (1 + 0,046) = 953 detik, sekitar 15,9 menit. Jeda mendominasi total waktu.
- **Latihan 6 dan 7:** perlu internet dan hasil crawl Anda, jadi isi hasilnya sendiri setelah sel dijalankan.

## O. Tugas Mandiri — Inverted Index dengan MapReduce
**Keputusan penanganan id kosong:** 5.577 judul kategori `inet` tidak punya id karena url-nya rusak. Baris itu **tidak dibuang** (satu kategori utuh akan hilang dari indeks). Setiap baris tanpa id diberi id pengganti berbentuk `inet-<nomor baris dalam partisi>` (string), sedangkan id asli tetap bilangan bulat. Id asli diurutkan lebih dulu, id pengganti sesudahnya. Keterbatasan: id pengganti hanya stabil selama urutan baris partisi tidak berubah, sehingga tidak boleh dipakai sebagai kunci join antarsistem.

In [ ]:
def mapper_indeks(path):
    """Mapper: (kata, id dokumen) untuk setiap kata unik pada judul."""
    df_ = pd.read_parquet(path, columns=["id", "judul"])
    kategori = path.parent.name.split("=")[1]
    hasil = []
    for no, (i, t) in enumerate(zip(df_["id"], df_["judul"]), start=1):
        id_dok = int(i) if pd.notna(i) else f"{kategori}-{no}"   # id pengganti bila url rusak
        for kata in set(tokenisasi(t)):
            hasil.append((kata, id_dok))
    return hasil

def kunci_urut(d):
    return (0, d) if isinstance(d, int) else (1, int(d.rsplit("-", 1)[1]))

# shuffle
shuffle_idx = collections.defaultdict(list)
for p in splits:
    for kata, id_dok in mapper_indeks(p):
        shuffle_idx[kata].append(id_dok)

# reduce: gabungkan id menjadi daftar terurut
indeks = {kata: sorted(ids, key=kunci_urut) for kata, ids in shuffle_idx.items()}
print("Jumlah kata di indeks:", f"{len(indeks):,}")
for kata in ("corona", "virus", "polisi"):
    print(f"{kata:<8} {len(indeks[kata]):>6,} judul | 5 id pertama: {indeks[kata][:5]}")

# pemeriksaan: jumlah id per kata = jumlah judul yang memuat kata itu
bersih_judul = pd.concat([pd.read_parquet(p, columns=["judul"]) for p in splits])["judul"]
cek_corona = sum(1 for t in bersih_judul if "corona" in tokenisasi(t))
print("\nJudul memuat 'corona' (hitung langsung):", f"{cek_corona:,}", "| sama dengan indeks:", cek_corona == len(indeks["corona"]))
n_pengganti = sum(1 for ids in indeks.values() for d in ids if isinstance(d, str))
print("Entri indeks dengan id pengganti (inet):", f"{n_pengganti:,}")

Jumlah kata di indeks: 32,275
corona   17,228 judul | 5 id pertama: [4856495, 4856598, 4868317, 4868545, 4869163]
virus     3,067 judul | 5 id pertama: [4856495, 4856598, 4860201, 4865102, 4865341]
polisi    2,724 judul | 5 id pertama: [4842273, 4842280, 4842541, 4842612, 4843641]

Judul memuat 'corona' (hitung langsung): 17,228 | sama dengan indeks: True
Entri indeks dengan id pengganti (inet): 43,634
